# CNN Shape versus Texture

**Nanthawat Sirisuksumran — Deep Learning**

ศึกษาผลของการผูกหรือสุ่มพื้นผิวต่อการตัดสินภาพเลขลายมือของ CNN โครงสร้างเดียวกัน ใช้ข้อมูลสาธารณะ UCI ผ่าน `load_digits` จำนวน 1,797 ภาพ แบ่ง train/validation/test ก่อนดัดแปลงภาพและฝึกโมเดลจริงจากศูนย์

**ผลอ้างอิงด้านล่างมาจากสคริปต์ที่รันจริงบน CPU** ไม่ใช่การอ้างว่า Notebook นี้ถูกรันใน Colab แล้ว ใช้ Runtime CPU และเลือก Run all เพื่อรันใหม่ โค้ดไม่ใช้ API key หรือโมเดลที่ดาวน์โหลดภายนอก

| Condition | Normal accuracy | Conflict shape | Conflict texture |
|---|---:|---:|---:|
| plain | 97.59% | 90.33% | 1.17% |
| correlated | 94.91% | 88.83% | 2.65% |
| randomized | 97.13% | 95.09% | 0.61% |

ผลหลัก: Randomized เพิ่ม conflict shape agreement 6.27 จุดเปอร์เซ็นต์จาก Correlated ภายใต้ข้อมูลและ CNN นี้ ไม่สรุปว่า CNN ทุกตัวใช้ global shape


## 1 ติดตั้งเวอร์ชันอ้างอิง

แนะนำให้ใช้ session ใหม่ Python 3.12 และ CPU เพื่อให้ตรงกับสภาพแวดล้อมอ้างอิง เวลารันจริงขึ้นอยู่กับเครื่อง


In [ ]:
# Reference execution uses CPU even when a GPU runtime is selected.
import os, sys, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = ''
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
    'torch==2.6.0+cpu', 'numpy==2.3.5', 'scipy==1.17.0',
    'scikit-learn==1.8.0', 'matplotlib==3.10.8', 'pillow==12.3.0'])
print('Dependencies installed. If this notebook previously imported other versions, restart the session before continuing.')


## 2 สร้างโครงการและแผนก่อนรัน

คำทำนายต้นฉบับอยู่ใน PREREGISTRATION.md และเก็บไว้ตามเดิม การเพิ่ม texture positive control เป็น exploratory หลังเห็นผลหลัก ตาม EXPLORATORY_ADDENDUM.md หากออกแบบการทดลองใหม่ ให้บันทึกคำทำนายใหม่ก่อนรัน


In [ ]:
import json
from pathlib import Path
project = Path.cwd() / 'shape_texture_project'
project.mkdir(exist_ok=True)
files = {'experiment.py': '"""Controlled shape/texture study. Train first; inspect test predictions afterward.\n\nCPU reference run: Python 3.12, PyTorch 2.6.0+cpu. No model/data download needed.\nSee PREREGISTRATION.md for hypotheses, controls, and limits of interpretation.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport sklearn\nimport torch\nfrom sklearn.datasets import load_digits\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.nn import functional as F\n\nROOT = Path(__file__).resolve().parent\nCONFIG = json.loads((ROOT / "config.json").read_text())\n\n\ndef save_json(path: Path, value) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef setup() -> None:\n    torch.set_num_threads(CONFIG["torch_threads"])\n    torch.set_num_interop_threads(1)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.mkldnn.deterministic = True\n    for folder in ("data", "results", "figures", "checkpoints"):\n        (ROOT / folder).mkdir(exist_ok=True)\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\nclass SmallCNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(1, 12, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),\n            nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n        )\n        self.classifier = nn.Sequential(\n            nn.Flatten(), nn.Linear(24 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, 10)\n        )\n\n    def forward(self, x):\n        return self.classifier(self.features(x))\n\n\nclass Renderer:\n    def __init__(self):\n        size = CONFIG["image_size"]\n        yy, xx = np.mgrid[:size, :size].astype(np.float64)\n        self.xx = (xx + 0.5) / size\n        self.yy = (yy + 0.5) / size\n        self.projections = []\n        for freq in CONFIG["texture_cycles_per_image"]:\n            for degrees in CONFIG["texture_angles_degrees"]:\n                angle = np.deg2rad(degrees)\n                self.projections.append(\n                    2 * np.pi * freq * (self.xx * np.cos(angle) + self.yy * np.sin(angle))\n                )\n        self.projections = np.asarray(self.projections)\n        self.disk = ((self.xx - 0.5) ** 2 + (self.yy - 0.5) ** 2) < 0.33 ** 2\n\n    def textured(self, masks, texture_labels, phases):\n        masks = np.asarray(masks, dtype=bool)\n        result = np.zeros(masks.shape, dtype=np.float32)\n        low, high = CONFIG["texture_range"]\n        # The tie-break noise is independent of digit and texture label. It only\n        # orders equal sine values; it does not change the prescribed histogram.\n        tie = np.random.default_rng(63031).uniform(0, 1e-9, (32, 32))\n        for i, (mask, texture, phase) in enumerate(zip(masks, texture_labels, phases)):\n            count = int(mask.sum())\n            assert count > 1\n            wave = np.sin(self.projections[int(texture)] + float(phase)) + tie\n            rank = np.argsort(np.argsort(wave[mask], kind="stable"), kind="stable")\n            result[i][mask] = np.linspace(low, high, count, dtype=np.float32)[rank]\n        return result\n\n\ndef get_data():\n    data = load_digits()\n    images = data.images.astype(np.float32) / 16.0\n    labels = data.target.astype(np.int64)\n    indices = np.arange(len(labels))\n    trainval, test = train_test_split(\n        indices, test_size=0.2, stratify=labels, random_state=CONFIG["split_seed"]\n    )\n    train, val = train_test_split(\n        trainval, test_size=0.25, stratify=labels[trainval], random_state=CONFIG["split_seed"]\n    )\n    splits = {"train": train, "validation": val, "test": test}\n    # Splitting happens before derivatives. Verify original indices and hashes.\n    hashes = [hashlib.sha256(a.tobytes()).hexdigest() for a in data.images]\n    for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):\n        assert not set(splits[a]) & set(splits[b])\n        assert not {hashes[i] for i in splits[a]} & {hashes[i] for i in splits[b]}, "Duplicate leakage"\n    normal = F.interpolate(\n        torch.from_numpy(images[:, None]), size=(32, 32), mode="bilinear", align_corners=False\n    ).numpy()[:, 0]\n    masks = normal > CONFIG["mask_threshold"]\n    split_info = {\n        k: {"count": len(v), "class_counts": np.bincount(labels[v], minlength=10).tolist(),\n            "indices": v.tolist()} for k, v in splits.items()\n    }\n    save_json(ROOT / "data/split.json", split_info)\n    np.savez_compressed(ROOT / "data/source_digits.npz", images=data.images.astype(np.uint8), labels=labels)\n    save_json(ROOT / "data/audit.json", {\n        "total_original_images": len(images), "unique_original_hashes": len(set(hashes)),\n        "cross_split_duplicate_count": 0, "split_before_transform": True,\n        "writer_disjoint": False,\n        "source_url": "https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits",\n        "loader_url": "https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html",\n        "license": "CC BY 4.0", "authors": "E. Alpaydin and C. Kaynak (1998)",\n        "doi": "10.24432/C50P49", "note": "1,797-image sklearn copy of UCI\'s original test subset; re-split for this study."\n    })\n    return normal, masks, labels, splits\n\n\ndef epoch_inputs(normal, masks, labels, condition, seed, epoch, renderer):\n    rng = np.random.default_rng(seed * 100_000 + epoch)\n    use_normal = rng.random(len(labels)) < CONFIG["normal_fraction"]\n    random_texture = rng.integers(0, 10, len(labels))\n    phases = rng.uniform(0, 2 * np.pi, len(labels))\n    if condition == "plain":\n        x = masks.astype(np.float32) * 0.6\n    else:\n        t = labels if condition == "correlated" else random_texture\n        x = renderer.textured(masks, t, phases)\n    x[use_normal] = normal[use_normal]\n    return torch.from_numpy(x[:, None])\n\n\n@torch.inference_mode()\ndef predict(model, images, return_logits=False):\n    model.eval()\n    x = torch.as_tensor(images, dtype=torch.float32)\n    if x.ndim == 3:\n        x = x[:, None]\n    logits = torch.cat([model(x[i:i + 256]).cpu() for i in range(0, len(x), 256)]).numpy()\n    return logits if return_logits else logits.argmax(axis=1)\n\n\ndef train_one(condition, seed, normal, masks, labels, splits, renderer):\n    checkpoint = ROOT / f"checkpoints/{condition}_seed{seed}.pt"\n    if checkpoint.exists():\n        stored = torch.load(checkpoint, map_location="cpu", weights_only=False)\n        assert stored["config"] == CONFIG, "Checkpoint/config mismatch"\n        print(f"REUSE {condition} seed={seed}", flush=True)\n        return stored["training_record"]\n    seed_everything(seed)\n    model = SmallCNN()\n    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"],\n                                 weight_decay=CONFIG["weight_decay"])\n    train, val = splits["train"], splits["validation"]\n    targets = torch.from_numpy(labels[train])\n    history = []\n    started = utc_now()\n    start_clock = time.perf_counter()\n    initial_hash = hashlib.sha256(b"".join(t.numpy().tobytes() for t in model.state_dict().values())).hexdigest()\n    for epoch in range(1, CONFIG["epochs"] + 1):\n        model.train()\n        x = epoch_inputs(normal[train], masks[train], labels[train], condition, seed, epoch, renderer)\n        order = np.random.default_rng(seed * 10_000 + epoch).permutation(len(train))\n        total_loss = total_correct = 0.0\n        for start in range(0, len(order), CONFIG["batch_size"]):\n            batch = order[start:start + CONFIG["batch_size"]]\n            optimizer.zero_grad(set_to_none=True)\n            output = model(x[batch])\n            loss = F.cross_entropy(output, targets[batch])\n            loss.backward()\n            optimizer.step()\n            total_loss += loss.item() * len(batch)\n            total_correct += (output.argmax(1) == targets[batch]).sum().item()\n        if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG["epochs"]:\n            val_logits = predict(model, normal[val], return_logits=True)\n            val_acc = float((val_logits.argmax(1) == labels[val]).mean())\n            val_loss = float(F.cross_entropy(torch.from_numpy(val_logits), torch.from_numpy(labels[val])))\n            history.append({"epoch": epoch, "train_loss": total_loss / len(train),\n                            "train_accuracy": total_correct / len(train),\n                            "normal_validation_accuracy": val_acc, "normal_validation_loss": val_loss})\n            print(f"{condition:10} seed={seed:2} epoch={epoch:2} train={total_correct/len(train):.3f} "\n                  f"val_normal={val_acc:.3f} elapsed={time.perf_counter()-start_clock:.1f}s", flush=True)\n    record = {"condition": condition, "seed": seed, "started_utc": started,\n              "finished_utc": utc_now(), "seconds": time.perf_counter() - start_clock,\n              "initial_state_sha256": initial_hash, "epochs": CONFIG["epochs"],\n              "parameter_count": sum(p.numel() for p in model.parameters()), "history": history,\n              "checkpoint_selection": "fixed_final_epoch; validation monitoring only"}\n    torch.save({"state_dict": model.state_dict(), "config": CONFIG, "training_record": record}, checkpoint)\n    save_json(ROOT / f"results/training_{condition}_seed{seed}.json", record)\n    return record\n\n\ndef evaluation_views(normal, masks, labels, test, renderer):\n    n = len(test)\n    y = labels[test]\n    rng = np.random.default_rng(CONFIG["evaluation_seed"])\n    # One phase for each source-image/texture pair; identical in aligned/conflict.\n    phases = rng.uniform(0, 2 * np.pi, (n, 10))\n    all_t = np.tile(np.arange(10), n)\n    all_images = renderer.textured(np.repeat(masks[test], 10, axis=0), all_t, phases.ravel())\n    match = all_t == np.repeat(y, 10)\n    conflict = ~match\n    views = {\n        "normal": (normal[test], y, np.full(n, -1)),\n        "plain": (masks[test].astype(np.float32) * 0.6, y, np.full(n, -1)),\n        "aligned": (all_images[match], y, y.copy()),\n        "conflict": (all_images[conflict], np.repeat(y, 10)[conflict], all_t[conflict]),\n        "texture_only_disk": (\n            renderer.textured(np.repeat(renderer.disk[None], n * 10, axis=0), all_t, phases.ravel()),\n            np.repeat(y, 10), all_t\n        ),\n    }\n    # Exact histogram equality across every replacement texture for each shape.\n    sorted_pixels = np.sort(all_images.reshape(n, 10, -1), axis=-1)\n    histogram_error = float(np.abs(sorted_pixels - sorted_pixels[:, :1]).max())\n    assert histogram_error == 0.0\n    save_json(ROOT / "results/render_audit.json", {\n        "paired_texture_max_sorted_pixel_difference": histogram_error,\n        "identical_foreground_masks": True, "conflicts_per_original_test_image": 9,\n        "test_original_count": n, "conflict_count": int(conflict.sum()),\n        "normal_images_untextured_grayscale": True,\n        "texture_id_definition": {str(k): {"cycles_per_image": CONFIG["texture_cycles_per_image"][k // 5],\n          "angle_degrees": CONFIG["texture_angles_degrees"][k % 5]} for k in range(10)}\n    })\n    np.savez_compressed(ROOT / "data/test_views.npz", **{k: v[0] for k, v in views.items()},\n                        digit_labels=y, test_indices=test, texture_ids=views["conflict"][2], phases=phases)\n    return views\n\n\ndef evaluate_all(views):\n    records = []\n    raw = {}\n    for condition in CONFIG["conditions"]:\n        for seed in CONFIG["seeds"]:\n            model = SmallCNN()\n            stored = torch.load(ROOT / f"checkpoints/{condition}_seed{seed}.pt", map_location="cpu", weights_only=False)\n            model.load_state_dict(stored["state_dict"])\n            raw[(condition, seed)] = {}\n            for name, (x, shape, texture) in views.items():\n                logits = predict(model, x, return_logits=True)\n                pred = logits.argmax(1)\n                record = {"condition": condition, "seed": seed, "view": name, "n": len(pred),\n                          "shape_accuracy": float(np.mean(pred == shape)),\n                          "texture_agreement": float(np.mean(pred == texture)) if np.all(texture >= 0) else None}\n                if name == "conflict":\n                    shape_match = pred == shape\n                    texture_match = pred == texture\n                    s, t = int(shape_match.sum()), int(texture_match.sum())\n                    record.update(shape_count=s, texture_count=t, other_count=len(pred)-s-t,\n                                  other_rate=float(1 - (s + t) / len(pred)),\n                                  coverage=float((s + t) / len(pred)),\n                                  conditional_shape_bias=float(s / (s+t)) if s+t else None)\n                records.append(record)\n                raw[(condition, seed)][name] = {"pred": pred, "shape": shape, "texture": texture}\n                np.savez_compressed(ROOT / f"results/predictions_{condition}_seed{seed}_{name}.npz",\n                                    logits=logits, prediction=pred, shape_label=shape, texture_label=texture)\n            print("EVALUATED", condition, seed, flush=True)\n    save_json(ROOT / "results/per_run_metrics.json", records)\n    columns = list(dict.fromkeys(k for row in records for k in row))\n    with (ROOT / "results/per_run_metrics.csv").open("w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=columns)\n        writer.writeheader()\n        writer.writerows(records)\n    summaries = []\n    for condition in CONFIG["conditions"]:\n        for view in views:\n            rows = [r for r in records if r["condition"] == condition and r["view"] == view]\n            result = {"condition": condition, "view": view, "n": rows[0]["n"]}\n            for key in ("shape_accuracy", "texture_agreement", "other_rate", "coverage", "conditional_shape_bias"):\n                if key not in rows[0] or rows[0][key] is None:\n                    continue\n                values = np.asarray([r[key] for r in rows], dtype=float)\n                result[key + "_mean"] = float(values.mean())\n                result[key + "_sd"] = float(values.std(ddof=1))\n            summaries.append(result)\n    save_json(ROOT / "results/summary.json", summaries)\n    # Pair all 9 conflicts belonging to the same original test image; first\n    # average across trained seeds. Bootstrap original images, never variants.\n    by_condition = {}\n    for condition in ("correlated", "randomized"):\n        sm = np.stack([raw[(condition, seed)]["conflict"]["pred"] == raw[(condition, seed)]["conflict"]["shape"]\n                       for seed in CONFIG["seeds"]]).reshape(len(CONFIG["seeds"]), -1, 9)\n        by_condition[condition] = sm.mean(axis=(0, 2))\n    per_image_delta = by_condition["randomized"] - by_condition["correlated"]\n    rng = np.random.default_rng(CONFIG["bootstrap_seed"])\n    bootstrap = np.asarray([per_image_delta[rng.integers(0, len(per_image_delta), len(per_image_delta))].mean()\n                            for _ in range(CONFIG["bootstrap_replicates"])])\n    seed_differences = []\n    for seed in CONFIG["seeds"]:\n        r = next(r for r in records if r["condition"] == "randomized" and r["seed"] == seed and r["view"] == "conflict")\n        c = next(r for r in records if r["condition"] == "correlated" and r["seed"] == seed and r["view"] == "conflict")\n        seed_differences.append(r["shape_accuracy"] - c["shape_accuracy"])\n    effect = {"contrast": "randomized_minus_correlated_conflict_shape_accuracy",\n              "mean_difference": float(per_image_delta.mean()),\n              "paired_seed_differences": seed_differences,\n              "paired_seed_difference_sd": float(np.std(seed_differences, ddof=1)),\n              "test_image_cluster_bootstrap_95ci": np.quantile(bootstrap, [0.025, 0.975]).tolist(),\n              "bootstrap_replicates": CONFIG["bootstrap_replicates"],\n              "interpretation": "CI conditional on fixed trained models and split; does not include writer/split/training uncertainty."}\n    save_json(ROOT / "results/paired_effect.json", effect)\n    print(json.dumps(summaries, indent=2), flush=True)\n    print("PAIRED_EFFECT", effect, flush=True)\n\n\ndef verify_reference():\n    """Re-evaluate every saved model and require identical predictions/logits."""\n    normal, masks, labels, splits = get_data()\n    views = evaluation_views(normal, masks, labels, splits["test"], Renderer())\n    checks = []\n    for condition in CONFIG["conditions"]:\n        for seed in CONFIG["seeds"]:\n            model = SmallCNN()\n            stored = torch.load(ROOT / f"checkpoints/{condition}_seed{seed}.pt", map_location="cpu", weights_only=False)\n            model.load_state_dict(stored["state_dict"])\n            for name, (x, _, _) in views.items():\n                reference = np.load(ROOT / f"results/predictions_{condition}_seed{seed}_{name}.npz")\n                logits = predict(model, x, return_logits=True)\n                assert np.array_equal(logits.argmax(1), reference["prediction"])\n                error = float(np.max(np.abs(logits-reference["logits"])))\n                assert error == 0, (condition, seed, name, error)\n                checks.append({"condition": condition, "seed": seed, "view": name, "max_logit_difference": error})\n    save_json(ROOT / "results/reproduction_check.json", {"checked_utc": utc_now(), "checks": checks,\n              "all_predictions_identical": True, "all_logits_identical": True,\n              "scope": "fresh model load and deterministic rendering/evaluation; not a second full training of every run"})\n    print(f"PASS: exact logits and predictions for all {len(checks)} model/view combinations")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--verify", action="store_true")\n    parser.add_argument("--train-only", action="store_true")\n    args = parser.parse_args()\n    setup()\n    if args.verify:\n        verify_reference()\n        return\n    reg = json.loads((ROOT / "preregistration_record.json").read_text())\n    assert sha256(ROOT / "PREREGISTRATION.md") == reg["protocol_sha256"]\n    assert sha256(ROOT / "config.json") == reg["config_sha256"]\n    environment = {"started_utc": utc_now(), "python": sys.version, "torch": torch.__version__,\n                   "numpy": np.__version__, "sklearn": sklearn.__version__, "platform": platform.platform(),\n                   "processor": platform.processor(), "torch_threads": torch.get_num_threads(),\n                   "device": "cpu", "config": CONFIG, "protocol_sha256": reg["protocol_sha256"],\n                   "experiment_source_sha256": sha256(Path(__file__))}\n    save_json(ROOT / "results/environment.json", environment)\n    normal, masks, labels, splits = get_data()\n    renderer = Renderer()\n    training_records = []\n    # No test evaluation occurs until every preregistered model has been trained.\n    for seed in CONFIG["seeds"]:\n        for condition in CONFIG["conditions"]:\n            training_records.append(train_one(condition, seed, normal, masks, labels, splits, renderer))\n    for seed in CONFIG["seeds"]:\n        initial = [r["initial_state_sha256"] for r in training_records if r["seed"] == seed]\n        assert len(set(initial)) == 1, "Initializations not paired"\n    save_json(ROOT / "results/training_records.json", training_records)\n    if not args.train_only:\n        views = evaluation_views(normal, masks, labels, splits["test"], renderer)\n        evaluate_all(views)\n    environment["finished_utc"] = utc_now()\n    save_json(ROOT / "results/environment.json", environment)\n\n\nif __name__ == "__main__":\n    main()\n', 'make_figures.py': '"""Create every report figure directly from the recorded run outputs."""\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / "figures"\nOUT.mkdir(exist_ok=True)\nplt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,\n                     "axes.spines.top": False, "axes.spines.right": False,\n                     "savefig.facecolor": "white", "figure.facecolor": "white"})\nCOLORS = {"plain": "#7B8894", "correlated": "#CB6443", "randomized": "#168278"}\nNAMES = {"plain": "Plain", "correlated": "Correlated", "randomized": "Randomized"}\n\n\ndef save(fig, name):\n    fig.savefig(OUT / f"{name}.png", dpi=240, bbox_inches="tight")\n    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight")\n    plt.close(fig)\n\n\ndef main():\n    data = np.load(ROOT / "data/test_views.npz")\n    labels = data["digit_labels"]\n    selected = [int(np.flatnonzero(labels == digit)[0]) for digit in (0,1,2,5,8)]\n    conflict = data["conflict"].reshape(len(labels),9,32,32)\n    texture_ids = data["texture_ids"].reshape(len(labels),9)\n    fig, axes = plt.subplots(3, 5, figsize=(7.1, 3.9))\n    for j, idx in enumerate(selected):\n        shape = int(labels[idx])\n        tex = (shape+3)%10\n        k = int(np.flatnonzero(texture_ids[idx]==tex)[0])\n        imgs = [data["normal"][idx], data["aligned"][idx], conflict[idx,k]]\n        titles = [f"shape {shape}", f"shape {shape} / texture {shape}", f"shape {shape} / texture {tex}"]\n        for i, (img,title) in enumerate(zip(imgs,titles)):\n            axes[i,j].imshow(img,cmap="gray",vmin=0,vmax=1,interpolation="nearest")\n            axes[i,j].set_xticks([]); axes[i,j].set_yticks([])\n            axes[i,j].set_title(title,fontsize=8,pad=4)\n        if j == 0:\n            for i, row in enumerate(("Normal", "Aligned", "Conflict")):\n                axes[i,j].set_ylabel(row,fontsize=10,labelpad=12)\n    fig.subplots_adjust(wspace=.18,hspace=.4,left=.1,right=.99,top=.93,bottom=.03)\n    save(fig,"test_examples")\n\n    fig,axes=plt.subplots(1,10,figsize=(7.1,1.15))\n    disks=data["texture_only_disk"][:10]\n    for t,ax in enumerate(axes):\n        ax.imshow(disks[t],cmap="gray",vmin=0,vmax=1,interpolation="nearest")\n        ax.set_title(f"{t}",fontsize=10,pad=3)\n        ax.set_xticks([]);ax.set_yticks([])\n        ax.set_xlabel(f"{(t%5)*36}deg\\n{3 if t<5 else 6} cyc",fontsize=6.5,labelpad=3)\n    fig.subplots_adjust(wspace=.12,left=.01,right=.99,top=.85,bottom=.25)\n    save(fig,"texture_bank")\n\n    summary=json.loads((ROOT/"results/summary.json").read_text())\n    get=lambda c,v:next(r for r in summary if r["condition"]==c and r["view"]==v)\n    fig,axes=plt.subplots(1,2,figsize=(7.1,2.9),gridspec_kw={"width_ratios":[1.1,1]})\n    conditions=["plain","correlated","randomized"]\n    x=np.arange(3)\n    for shift,view,label in [(-.2,"normal","Normal"),(.2,"aligned","Aligned")]:\n        y=[100*get(c,view)["shape_accuracy_mean"] for c in conditions]\n        sd=[100*get(c,view)["shape_accuracy_sd"] for c in conditions]\n        axes[0].bar(x+shift,y,width=.36,label=label,color="#8E9BAC" if view=="normal" else "#445A72",\n                    yerr=sd,capsize=2,error_kw={"linewidth":.8})\n        for a,b in zip(x+shift,y):axes[0].text(a,b-6,f"{b:.1f}",ha="center",va="top",color="white",fontsize=8)\n    axes[0].set_xticks(x,[NAMES[c] for c in conditions],fontsize=8)\n    axes[0].set_ylim(0,105);axes[0].set_ylabel("Digit accuracy (%)")\n    axes[0].set_title("Competence checks",fontsize=11)\n    axes[0].legend(frameon=False,loc="upper center",bbox_to_anchor=(.5,-.16),ncol=2,fontsize=8)\n    bottom=np.zeros(3)\n    for key,label,color in [("shape_accuracy_mean","Shape","#168278"),\n                            ("texture_agreement_mean","Texture","#CB6443"),\n                            ("other_rate_mean","Other","#D7DCE2")]:\n        values=np.array([100*get(c,"conflict")[key] for c in conditions])\n        axes[1].bar(x,values,bottom=bottom,width=.58,label=label,color=color)\n        if label=="Shape":\n            for a,b in zip(x,values):axes[1].text(a,b/2,f"{b:.1f}",ha="center",va="center",color="white",fontsize=9)\n        bottom+=values\n    axes[1].set_xticks(x,[NAMES[c] for c in conditions],fontsize=8)\n    axes[1].set_ylim(0,105);axes[1].set_ylabel("All conflict predictions (%)")\n    axes[1].set_title("Cue-conflict decisions",fontsize=11)\n    axes[1].legend(frameon=False,loc="upper center",bbox_to_anchor=(.5,-.16),ncol=3,fontsize=7)\n    fig.tight_layout(w_pad=1.7)\n    save(fig,"main_results")\n\n    histories=json.loads((ROOT/"results/training_records.json").read_text())\n    fig,axes=plt.subplots(1,2,figsize=(7.1,2.6))\n    for c in conditions:\n        records=[r for r in histories if r["condition"]==c]\n        epochs=[r["epoch"] for r in records[0]["history"]]\n        for key,ax in [("train_loss",axes[0]),("normal_validation_accuracy",axes[1])]:\n            values=np.array([[e[key] for e in r["history"]] for r in records])\n            mean=values.mean(0);sd=values.std(0,ddof=1)\n            if "accuracy" in key:mean*=100;sd*=100\n            ax.plot(epochs,mean,color=COLORS[c],label=NAMES[c],linewidth=1.6)\n            ax.fill_between(epochs,mean-sd,mean+sd,color=COLORS[c],alpha=.13)\n    axes[0].set_ylabel("Training cross entropy");axes[0].set_xlabel("Epoch")\n    axes[1].set_ylabel("Normal validation accuracy (%)");axes[1].set_xlabel("Epoch")\n    axes[1].set_ylim(20,100);axes[0].legend(frameon=False,fontsize=8)\n    fig.tight_layout()\n    save(fig,"training_curves")\n\n    # Deterministic example of a decision change; explicitly illustrative rather\n    # than a random sample. Main metrics always use every held-out original.\n    cor=np.load(ROOT/"results/predictions_correlated_seed11_conflict.npz")\n    rnd=np.load(ROOT/"results/predictions_randomized_seed11_conflict.npz")\n    changed=np.flatnonzero((cor["prediction"]==cor["texture_label"]) &\n                          (rnd["prediction"]==rnd["shape_label"]))\n    selected_changes=changed[:4]\n    fig,axes=plt.subplots(1,4,figsize=(6.7,2.15))\n    for ax,k in zip(axes,selected_changes):\n        ax.imshow(data["conflict"][k],cmap="gray",vmin=0,vmax=1,interpolation="nearest")\n        s,t=int(cor["shape_label"][k]),int(cor["texture_label"][k])\n        ax.set_title(f"Shape {s} / texture {t}",fontsize=9,pad=5)\n        ax.set_xlabel(f"Correlated: {int(cor[\'prediction\'][k])}\\nRandomized: {int(rnd[\'prediction\'][k])}",fontsize=9)\n        ax.set_xticks([]);ax.set_yticks([])\n    fig.tight_layout(w_pad=1)\n    save(fig,"illustrative_decisions")\n    print("created figures",*[p.name for p in sorted(OUT.glob(\'*.png\'))])\n\n\nif __name__=="__main__":\n    main()\n', 'positive_control.py': '"""Post-primary, explicitly exploratory texture-learnability control."""\nimport hashlib\nimport json\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nimport experiment as ex\n\n\ndef main():\n    ex.setup()\n    declaration = ex.ROOT / "EXPLORATORY_ADDENDUM.md"\n    marker = ex.ROOT / "exploratory_record.json"\n    if not marker.exists():\n        ex.save_json(marker, {"recorded_before_extra_training_utc": ex.utc_now(),\n                             "addendum_sha256": hashlib.sha256(declaration.read_bytes()).hexdigest(),\n                             "primary_results_already_observed": True})\n    renderer = ex.Renderer()\n    n = 1077\n    masks = np.repeat(renderer.disk[None], n, axis=0)\n    targets = np.arange(n) % 10\n    y = torch.from_numpy(targets)\n    views = np.load(ex.ROOT / "data/test_views.npz")\n    test_x = views["texture_only_disk"]\n    test_y = np.tile(np.arange(10), len(test_x) // 10)\n    output = []\n    for seed in ex.CONFIG["seeds"]:\n        ex.seed_everything(seed)\n        model = ex.SmallCNN()\n        optimizer = torch.optim.Adam(model.parameters(), lr=ex.CONFIG["learning_rate"],\n                                     weight_decay=ex.CONFIG["weight_decay"])\n        started = ex.utc_now()\n        clock = time.perf_counter()\n        for epoch in range(1, ex.CONFIG["epochs"] + 1):\n            model.train()\n            rng = np.random.default_rng(33_000_000 + seed * 100_000 + epoch)\n            phase = rng.uniform(0, 2*np.pi, n)\n            x = torch.from_numpy(renderer.textured(masks, targets, phase)[:, None])\n            order = rng.permutation(n)\n            for a in range(0, n, ex.CONFIG["batch_size"]):\n                idx = order[a:a+ex.CONFIG["batch_size"]]\n                optimizer.zero_grad(set_to_none=True)\n                loss = F.cross_entropy(model(x[idx]), y[idx])\n                loss.backward()\n                optimizer.step()\n            if epoch % 10 == 0:\n                print("texture_positive_control", seed, epoch, float(loss), flush=True)\n        logits = ex.predict(model, test_x, return_logits=True)\n        pred = logits.argmax(1)\n        result = {"seed": seed, "n": len(test_y), "accuracy": float((pred==test_y).mean()),\n                  "started_utc": started, "finished_utc": ex.utc_now(), "seconds": time.perf_counter()-clock,\n                  "status": "exploratory added after primary results"}\n        torch.save({"state_dict": model.state_dict(), "config": ex.CONFIG, "result": result},\n                   ex.ROOT/f"checkpoints/texture_positive_control_seed{seed}.pt")\n        np.savez_compressed(ex.ROOT/f"results/positive_control_seed{seed}.npz",\n                            logits=logits,prediction=pred,target=test_y)\n        output.append(result)\n        print(result,flush=True)\n    ex.save_json(ex.ROOT/"results/positive_control.json", {\n        "status": "exploratory added after primary results",\n        "accuracy_mean": float(np.mean([r["accuracy"] for r in output])),\n        "accuracy_sd": float(np.std([r["accuracy"] for r in output],ddof=1)), "runs": output})\n\n\nif __name__ == "__main__":\n    main()\n', 'verify_retraining.py': '"""Repeat all nine trainings in a temporary directory and compare exact states.\n\nOriginal reference outputs are never overwritten. This is stronger than merely\nreloading saved checkpoints. It is expected to match exactly in the reference\nCPU environment; different PyTorch/CPU versions need not be bitwise identical.\n"""\nimport json\nimport shutil\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nimport experiment as ex\n\n\ndef main():\n    reference = ex.ROOT\n    ex.setup()\n    with tempfile.TemporaryDirectory(prefix="retraining_", dir=reference) as folder:\n        ex.ROOT = Path(folder)\n        for name in ("config.json", "PREREGISTRATION.md", "preregistration_record.json"):\n            shutil.copy2(reference / name, ex.ROOT / name)\n        for name in ("data", "results", "checkpoints", "figures"):\n            (ex.ROOT / name).mkdir()\n        normal, masks, labels, splits = ex.get_data()\n        renderer = ex.Renderer()\n        checks = []\n        for seed in ex.CONFIG["seeds"]:\n            for condition in ex.CONFIG["conditions"]:\n                ex.train_one(condition, seed, normal, masks, labels, splits, renderer)\n                filename = f"{condition}_seed{seed}.pt"\n                ref = torch.load(reference / "checkpoints" / filename, weights_only=False, map_location="cpu")\n                new = torch.load(ex.ROOT / "checkpoints" / filename, weights_only=False, map_location="cpu")\n                errors = {k: float(torch.max(torch.abs(v-new["state_dict"][k]))) for k,v in ref["state_dict"].items()}\n                assert all(v == 0 for v in errors.values()), (condition, seed, errors)\n                checks.append({"condition": condition, "seed": seed, "max_parameter_difference": max(errors.values())})\n                print("EXACT RETRAINING MATCH", condition, seed, flush=True)\n        views = ex.evaluation_views(normal, masks, labels, splits["test"], renderer)\n        ex.evaluate_all(views)\n        assert json.loads((reference/"results/per_run_metrics.json").read_text()) == json.loads((ex.ROOT/"results/per_run_metrics.json").read_text())\n        prediction_checks = 0\n        for filename in (reference/"results").glob("predictions_*.npz"):\n            a = np.load(filename)\n            b = np.load(ex.ROOT/"results"/filename.name)\n            assert all(np.array_equal(a[k],b[k]) for k in a.files), filename.name\n            prediction_checks += 1\n        result = {"checked_utc": ex.utc_now(), "full_retraining_runs": len(checks),\n                  "all_weights_identical": True, "all_metrics_identical": True,\n                  "all_logits_and_predictions_identical": True,\n                  "prediction_files_compared": prediction_checks, "checks": checks,\n                  "scope": "same pinned reference environment, fixed CPU threads and data split"}\n        ex.save_json(reference/"results/full_retraining_check.json", result)\n        print("PASS: all nine independent re-trainings exactly reproduce weights, logits, and metrics",flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'config.json': '{\n  "protocol_version": "1.0",\n  "split_seed": 20261031,\n  "seeds": [11, 22, 33],\n  "conditions": ["plain", "correlated", "randomized"],\n  "epochs": 50,\n  "batch_size": 128,\n  "learning_rate": 0.001,\n  "weight_decay": 0.0001,\n  "image_size": 32,\n  "mask_threshold": 0.25,\n  "normal_fraction": 0.25,\n  "texture_angles_degrees": [0, 36, 72, 108, 144],\n  "texture_cycles_per_image": [3, 6],\n  "texture_range": [0.25, 0.95],\n  "torch_threads": 2,\n  "evaluation_seed": 71031,\n  "bootstrap_seed": 91031,\n  "bootstrap_replicates": 2000\n}\n', 'PREREGISTRATION.md': "# Predictions and locked experimental plan (recorded before training)\n\nThis is an AI-assisted prediction and study design, written before any CNN was\ntrained or any model prediction was inspected. It is a local audit record, not\nan independently registered or externally timestamped preregistration.\n\n## Research question\nFor the same small CNN, does changing the relationship between a digit's shape\nlabel and a synthetic foreground texture change its decisions under cue conflict?\n\n## Predictions, to be retained even if wrong\n1. With textures correlated with digit labels, the CNN will follow texture more\n   often than a CNN trained with texture independent of digit labels.\n2. Randomizing textures during training will increase conflict shape agreement\n   and conditional shape bias, while preserving useful accuracy on normal digits.\n3. The plain-digit baseline will follow digit shape on most conflict images.\n4. All three conditions will achieve at least 80% normal test accuracy. This is\n   a declared interpretation safeguard, not a threshold imposed by the assignment.\n\n## Data and controls\nUse all 1,797 images from sklearn.datasets.load_digits (UCI Optical Recognition\nof Handwritten Digits, 10 classes). Split original-image indices into 60% train,\n20% validation, 20% test, stratified by digit with random_state=20261031. All\nderivatives of an original image remain in its split. Inspect exact duplicate\nsource-image hashes for leakage. No claim of writer-disjoint splits is made.\n\nUpscale 8x8 images to 32x32 with bilinear interpolation. Preserve the unmodified\ngrayscale version for normal evaluation. Define a foreground mask at intensity\n>0.25. Texture is a sinusoidal grating: five orientations times two frequencies,\nwith randomized phase. Rank-map foreground values onto the same evenly spaced\nintensity sequence (0.25..0.95) for each mask. Thus paired cue-conflict images\nhave exactly the same mask, background, and foreground intensity histogram.\nTexture spatial arrangement changes; color, pixel count, brightness, and\nhistogram cannot identify the replacement texture within a paired shape.\n\n## Intervention and training\nTrain the identical CNN from scratch in three conditions: plain (untextured\ndigits), correlated (texture class equals shape label), randomized (texture\nindependent of shape label). In every condition, the same 25% of examples per\nepoch are normal grayscale digits. The remaining 75% are flat foreground\ndigits in plain or textured digits in the other conditions. Randomized texture\nlabels are sampled independently, not guaranteed perfectly balanced in any\nsingle epoch. All conditions use identical source images, epoch count, optimizer,\nbatch order, normal-example flags, texture phases, and initialization for a paired\nseed; only the treatment-dependent rendering changes. Seeds: 11,22,33.\nUse the fixed final epoch (50), not the best conflict/test result. Validation is\nmonitoring only; no hyperparameter or epoch selection from test results. Preserve\nall runs and report failure if the normal-accuracy safeguard is not satisfied.\n\n## Evaluation\nEvaluate identical held-out images for every model: normal grayscale; plain\nmask; aligned texture; all 9 mismatching texture labels per digit (exhaustive\ncue conflict); and texture-only fixed disk. Reuse one deterministic test phase\nper source-image/texture pair. The disk is a supplementary, out-of-distribution\ndiagnostic, not the main evidence for texture use.\n\nFor conflicts let S=predictions matching shape, T=matching texture, O=other.\nReport S/N, T/N, O/N and conditional shape bias S/(S+T), not just conditional\nshape bias. Report coverage (S+T)/N. Chance digit accuracy is 10%; conditional\nshape bias has chance expectation 50% when the two cues are exchangeable.\nReport all seeds and mean +/- sample SD across seeds. Compare randomized minus\ncorrelated using paired seeds. For a conditional 95% CI, average model outcomes\nacross seeds first, then cluster-bootstrap original test images with their 9\nconflicts together (2,000 draws). This CI describes test-image uncertainty\nconditional on this split and these trained models, not population training or\nwriter uncertainty. Do not treat transformed variants as independent images.\n\n## Limitations to report\nSynthetic gratings on small binarized handwritten digits are not natural object\ntextures. Digit shape includes local strokes; shape agreement does not prove use\nof global outline rather than local shape. The experiment measures a learned\ntraining-data dependence in this CNN and domain, not an inherent property of all\nCNNs. The normal/texture mixture, foreground masking, and fixed-disk diagnostic\nproduce distribution shifts. One split and three training seeds do not establish\nwriter-level generalization. No accuracy drop alone proves a specific cue.\n", 'preregistration_record.json': '{\n  "recorded_before_training_utc": "2026-10-31T09:27:16.852327+00:00",\n  "protocol_sha256": "8f92113defff8c79497bb08f64800247db1791b00f0ee2eb3895ea0a69887a2f",\n  "config_sha256": "7a41cd4e54a6f7fb5f6ec66f7c9d70af578ac126a3151c70cfa78cc43ec46150"\n}', 'EXPLORATORY_ADDENDUM.md': '# Exploratory positive control added after the primary results\n\nThe primary run showed exactly 10% texture agreement for the fixed-disk\nout-of-distribution diagnostic. That result alone cannot distinguish learned\nshape dependence from failure to recognize unfamiliar disk inputs, nor establish\nthat the synthetic texture bank is learnable. The following additional diagnostic\nis therefore declared after observing the primary results and before training\nthe extra models. It does not replace, select, or alter any primary experiment.\n\nTrain the same 101,946-parameter CNN on fixed disks filled with the same ten\ntextures, with texture identity as the target. Use 1,077 training samples per\nepoch, class labels cycling 0..9, random phase per epoch, the same three seeds,\nAdam settings, batch size and fixed 50 epochs. Evaluate the same 3,600 fixed-disk\ntest images used in the primary evaluation. The phases are deterministic but\nindependent from the training generator. Predict that test accuracy will exceed\n80%. Report this control as exploratory, even if it fails. It only establishes\nlearnability of the texture bank by the architecture under texture supervision;\nit does not prove that a primary model used or ignored any specific cue.\n', 'exploratory_record.json': '{\n  "recorded_before_extra_training_utc": "2026-10-31T09:32:21.215298+00:00",\n  "addendum_sha256": "f2e5f324df8d423f37e1041cb8cd867fff5ea48adba244bf1a36601db44ee296",\n  "primary_results_already_observed": true\n}', 'requirements.txt': '--extra-index-url https://download.pytorch.org/whl/cpu\ntorch==2.6.0+cpu\nnumpy==2.3.5\nscipy==1.17.0\nscikit-learn==1.8.0\nmatplotlib==3.10.8\npillow==12.3.0\n'}
for name, content in files.items():
    (project / name).write_text(content, encoding='utf-8')
(project / 'REFERENCE_METRICS.json').write_text('[\n  {\n    "condition": "plain",\n    "seed": 11,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9777777777777777,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 11,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9694444444444444,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 11,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9055555555555556,\n    "texture_agreement": 0.9055555555555556\n  },\n  {\n    "condition": "plain",\n    "seed": 11,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.9145061728395062,\n    "texture_agreement": 0.011111111111111112,\n    "shape_count": 2963,\n    "texture_count": 36,\n    "other_count": 241,\n    "other_rate": 0.07438271604938274,\n    "coverage": 0.9256172839506173,\n    "conditional_shape_bias": 0.9879959986662221\n  },\n  {\n    "condition": "plain",\n    "seed": 11,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "plain",\n    "seed": 22,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9722222222222222,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 22,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9555555555555556,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 22,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.8916666666666667,\n    "texture_agreement": 0.8916666666666667\n  },\n  {\n    "condition": "plain",\n    "seed": 22,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.8824074074074074,\n    "texture_agreement": 0.012654320987654321,\n    "shape_count": 2859,\n    "texture_count": 41,\n    "other_count": 340,\n    "other_rate": 0.1049382716049383,\n    "coverage": 0.8950617283950617,\n    "conditional_shape_bias": 0.9858620689655172\n  },\n  {\n    "condition": "plain",\n    "seed": 22,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "plain",\n    "seed": 33,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9777777777777777,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 33,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9777777777777777,\n    "texture_agreement": null\n  },\n  {\n    "condition": "plain",\n    "seed": 33,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9166666666666666,\n    "texture_agreement": 0.9166666666666666\n  },\n  {\n    "condition": "plain",\n    "seed": 33,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.912962962962963,\n    "texture_agreement": 0.011419753086419753,\n    "shape_count": 2958,\n    "texture_count": 37,\n    "other_count": 245,\n    "other_rate": 0.07561728395061729,\n    "coverage": 0.9243827160493827,\n    "conditional_shape_bias": 0.9876460767946578\n  },\n  {\n    "condition": "plain",\n    "seed": 33,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "correlated",\n    "seed": 11,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9555555555555556,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 11,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9416666666666667,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 11,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9833333333333333,\n    "texture_agreement": 0.9833333333333333\n  },\n  {\n    "condition": "correlated",\n    "seed": 11,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.8873456790123457,\n    "texture_agreement": 0.03271604938271605,\n    "shape_count": 2875,\n    "texture_count": 106,\n    "other_count": 259,\n    "other_rate": 0.07993827160493827,\n    "coverage": 0.9200617283950617,\n    "conditional_shape_bias": 0.9644414625964441\n  },\n  {\n    "condition": "correlated",\n    "seed": 11,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "correlated",\n    "seed": 22,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9638888888888889,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 22,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9361111111111111,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 22,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9861111111111112,\n    "texture_agreement": 0.9861111111111112\n  },\n  {\n    "condition": "correlated",\n    "seed": 22,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.9030864197530865,\n    "texture_agreement": 0.022530864197530864,\n    "shape_count": 2926,\n    "texture_count": 73,\n    "other_count": 241,\n    "other_rate": 0.07438271604938274,\n    "coverage": 0.9256172839506173,\n    "conditional_shape_bias": 0.9756585528509503\n  },\n  {\n    "condition": "correlated",\n    "seed": 22,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "correlated",\n    "seed": 33,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9277777777777778,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 33,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.8777777777777778,\n    "texture_agreement": null\n  },\n  {\n    "condition": "correlated",\n    "seed": 33,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9694444444444444,\n    "texture_agreement": 0.9694444444444444\n  },\n  {\n    "condition": "correlated",\n    "seed": 33,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.8743827160493827,\n    "texture_agreement": 0.024382716049382715,\n    "shape_count": 2833,\n    "texture_count": 79,\n    "other_count": 328,\n    "other_rate": 0.10123456790123453,\n    "coverage": 0.8987654320987655,\n    "conditional_shape_bias": 0.9728708791208791\n  },\n  {\n    "condition": "correlated",\n    "seed": 33,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "randomized",\n    "seed": 11,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9583333333333334,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 11,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9583333333333334,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 11,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9416666666666667,\n    "texture_agreement": 0.9416666666666667\n  },\n  {\n    "condition": "randomized",\n    "seed": 11,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.9441358024691358,\n    "texture_agreement": 0.006172839506172839,\n    "shape_count": 3059,\n    "texture_count": 20,\n    "other_count": 161,\n    "other_rate": 0.04969135802469138,\n    "coverage": 0.9503086419753086,\n    "conditional_shape_bias": 0.9935043845404352\n  },\n  {\n    "condition": "randomized",\n    "seed": 11,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "randomized",\n    "seed": 22,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9777777777777777,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 22,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.975,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 22,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.9527777777777777,\n    "texture_agreement": 0.9527777777777777\n  },\n  {\n    "condition": "randomized",\n    "seed": 22,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.9537037037037037,\n    "texture_agreement": 0.006172839506172839,\n    "shape_count": 3090,\n    "texture_count": 20,\n    "other_count": 130,\n    "other_rate": 0.04012345679012341,\n    "coverage": 0.9598765432098766,\n    "conditional_shape_bias": 0.9935691318327974\n  },\n  {\n    "condition": "randomized",\n    "seed": 22,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  },\n  {\n    "condition": "randomized",\n    "seed": 33,\n    "view": "normal",\n    "n": 360,\n    "shape_accuracy": 0.9777777777777777,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 33,\n    "view": "plain",\n    "n": 360,\n    "shape_accuracy": 0.9638888888888889,\n    "texture_agreement": null\n  },\n  {\n    "condition": "randomized",\n    "seed": 33,\n    "view": "aligned",\n    "n": 360,\n    "shape_accuracy": 0.95,\n    "texture_agreement": 0.95\n  },\n  {\n    "condition": "randomized",\n    "seed": 33,\n    "view": "conflict",\n    "n": 3240,\n    "shape_accuracy": 0.9549382716049383,\n    "texture_agreement": 0.005864197530864198,\n    "shape_count": 3094,\n    "texture_count": 19,\n    "other_count": 127,\n    "other_rate": 0.03919753086419753,\n    "coverage": 0.9608024691358025,\n    "conditional_shape_bias": 0.9938965628011565\n  },\n  {\n    "condition": "randomized",\n    "seed": 33,\n    "view": "texture_only_disk",\n    "n": 3600,\n    "shape_accuracy": 0.1,\n    "texture_agreement": 0.1\n  }\n]', encoding='utf-8')
print('Project created:', project)


In [ ]:
print((project / 'PREREGISTRATION.md').read_text())


## 3 ฝึก CNN และประเมินผลหลัก

ฝึก 3 เงื่อนไข × 3 seed × 50 epochs เลือก fixed final epoch ไม่เลือกจาก test หลังฝึกครบทุกโมเดลจึงประเมินภาพทดสอบ หากรันซ้ำในโฟลเดอร์เดิมจะ reuse checkpoints ใช้เซลล์ตรวจ full retraining ด้านล่างเพื่อตรวจการฝึกใหม่


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-u', 'experiment.py'], cwd=project)


## 4 ผล ตัวเลข และภาพทดสอบจริง

S คือคำตอบตามรูปเลข T คือคำตอบตาม texture ID O คือคลาสอื่น Conditional shape bias = S/(S+T) จึงต้องดู S/N, T/N, O/N และ coverage ด้วย CI ใช้ bootstrap ที่ระดับต้นฉบับ 360 ภาพ ไม่ถือภาพแปลงทั้ง 3,240 ภาพเป็นอิสระ


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(project / 'results/per_run_metrics.csv'))
display(pd.DataFrame(json.loads((project / 'results/summary.json').read_text())))
print((project / 'results/paired_effect.json').read_text())
subprocess.check_call([sys.executable, 'make_figures.py'], cwd=project)
for name in ['test_examples', 'texture_bank', 'main_results', 'training_curves', 'illustrative_decisions']:
    display(Image(filename=str(project / 'figures' / (name+'.png'))))


## 5 ตรวจเทียบผลอ้างอิง

ตัวเลขอ้างอิงเป็นผลที่บันทึกจากการรันจริง ไม่ได้สร้างเพื่อให้ตรงสมมติฐาน ต่างเวอร์ชันหรือ CPU อาจมีความต่างเชิงตัวเลข จึงแสดงความต่างอย่างเปิดเผย


In [ ]:
reference = json.loads((project / 'REFERENCE_METRICS.json').read_text())
current = json.loads((project / 'results/per_run_metrics.json').read_text())
print('All reference metrics exactly match:', current == reference)
if current != reference:
    print('Environment-dependent differences found; retain current values and do not claim an exact match.')
subprocess.check_call([sys.executable, 'experiment.py', '--verify'], cwd=project)


## 6 ตรวจการฝึกใหม่ทั้งเก้าโมเดล

เซลล์นี้ใช้เวลาฝึกเพิ่ม รันในโฟลเดอร์ชั่วคราวแล้วเปรียบเทียบน้ำหนัก logits predictions และ metrics ทั้งหมด ผลอ้างอิงผ่านการตรวจนี้แล้วโดยตรงกันทุกค่าในสภาพแวดล้อมเดิม


In [ ]:
subprocess.check_call([sys.executable, '-u', 'verify_retraining.py'], cwd=project)
print((project / 'results/full_retraining_check.json').read_text())


## 7 การตรวจเพิ่มหลังผลหลัก

โมเดลหลักตอบ texture บน disk ได้ 10% แต่เป็น out of distribution จึงเพิ่มการฝึก CNN บน disk โดยตรงเพื่อทดสอบว่า texture bank เรียนรู้ได้หรือไม่ ผลอ้างอิง 100% ทั้งสาม seed การตรวจนี้เป็น exploratory และไม่เปลี่ยนผลหรือคำทำนายหลัก


In [ ]:
print((project / 'EXPLORATORY_ADDENDUM.md').read_text())
subprocess.check_call([sys.executable, '-u', 'positive_control.py'], cwd=project)
print((project / 'results/positive_control.json').read_text())


## 8 ดาวน์โหลดผลทั้งหมด


In [ ]:
import shutil
archive = shutil.make_archive(str(project.parent / 'Shape_Texture_Run'), 'zip', root_dir=project.parent, base_dir=project.name)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Saved:', archive)


## ข้อจำกัดและการใช้ AI

ข้อมูลเป็นตัวเลขขนาดเล็กและลายสังเคราะห์ มีหนึ่ง split และสาม seed ไม่แยกตามผู้เขียน Shape agreement อาจมาจาก local strokes ไม่จำเป็นต้องเป็น global outline การดีขึ้นอาจรวมความทนต่อภาพดัดแปลง การลบรูปร่างด้วย disk ทำให้ distribution เปลี่ยน CI ที่รายงานสะท้อนความไม่แน่นอนของภาพทดสอบภายใต้โมเดลและ split คงที่

AI ช่วยตั้งสมมติฐาน ออกแบบ เขียนและรันโค้ด วิเคราะห์ ตรวจซ้ำ และร่างรายงาน ผู้ส่งควรอ่านและทำความเข้าใจงานทั้งหมดก่อนส่ง

Dataset: Alpaydin & Kaynak (1998), UCI Optical Recognition of Handwritten Digits, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), [DOI 10.24432/C50P49](https://doi.org/10.24432/C50P49). ใช้ subset 1,797 ภาพที่ bundled ใน scikit-learn ไม่ใช่ UCI ทั้งชุด 5,620 ภาพ

Background: [Geirhos et al. 2019](https://arxiv.org/abs/1811.12231); [Burgert et al. 2025](https://arxiv.org/abs/2509.20234).
